# Synthetic data validation

Generate the synthetic D2C panel and check that:
1. The dataset has the expected shape, calendar features and outcome distribution.
2. The Ridge MMM recovers the **ground-truth** channel coefficients (the credibility moment).
3. Decomposition of fitted units sums to the prediction (sanity).

In [ ]:
import sys
from pathlib import Path

# Make src importable when running this notebook from the repo root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.express as px

from src.data_generation import CHANNELS, generate_dataset
from src.mmm import MMM

## 1. Generate the panel

In [ ]:
df, truth = generate_dataset(seed=42)
print(f'Rows: {len(df):,}  ({df.date.min().date()} to {df.date.max().date()})')
print(f'Columns: {df.shape[1]}')
df.head()

In [ ]:
df[['units_sold', 'revenue_chf']].describe().round(0)

## 2. Calendar / control flags

In [ ]:
summary = pd.DataFrame({
    'is_weekend share':   [df.is_weekend.mean()],
    'is_holiday share':   [df.is_holiday.mean()],
    'temperature mean':   [df.temperature_c.mean()],
    'temperature min':    [df.temperature_c.min()],
    'temperature max':    [df.temperature_c.max()],
    'promo_pct mean':     [df.promo_pct.mean()],
}).T.round(2).rename(columns={0: 'value'})
summary

## 3. Marketing share by channel

In [ ]:
contrib_cols = [c for c in df.columns if c.startswith('contrib_')]
share = (df[contrib_cols].sum() / df['units_sold'].sum()).sort_values(ascending=False)
share.to_frame('share of units').style.format('{:.2%}')

## 4. TV burst window — May 2025

In [ ]:
burst = df[(df.date >= '2025-05-05') & (df.date < '2025-06-02')]
control_period = df[(df.date >= '2024-05-05') & (df.date < '2024-06-02')]
print(f'TV burst weeks       : avg units = {burst.units_sold.mean():,.0f}')
print(f'Same period 2024 (no burst): avg units = {control_period.units_sold.mean():,.0f}')
print(f'Naive lift            : {burst.units_sold.mean() - control_period.units_sold.mean():,.0f}')
px.line(df.assign(year=df.date.dt.year), x='date', y='tv_spend',
        title='TV spend over time — burst clearly visible')

## 5. Ground-truth recovery — fit MMM and compare

In [ ]:
mmm = MMM()
fit = mmm.fit(df)
print(f'R² = {fit.r_squared:.3f}  ·  MAPE = {fit.mape:.2%}')

In [ ]:
comparison = pd.DataFrame({
    'true β':       [truth.betas[c.name]    for c in CHANNELS],
    'estimated β':  [fit.betas[c.name]      for c in CHANNELS],
    'true λ':       [truth.decay[c.name]    for c in CHANNELS],
    'estimated λ':  [fit.decay[c.name]      for c in CHANNELS],
    'true k':       [truth.half_sat[c.name] for c in CHANNELS],
    'estimated k':  [fit.half_sat[c.name]   for c in CHANNELS],
}, index=[c.name for c in CHANNELS])
comparison.style.format('{:,.2f}')

Search and social are the dominant always-on channels — Ridge fallback recovers their coefficients well. TV is sparse + bursty so its β is harder to identify with a single linear fit; the Bayesian PyMC version (notebook 02) handles this better with informative priors.

## 6. Decomposition sanity check

In [ ]:
rebuilt = (
    fit.contribution['baseline']
    + fit.contribution[[c for c in fit.contribution.columns if c.startswith('contrib_')]].sum(axis=1)
).to_numpy()
max_diff = float(np.max(np.abs(rebuilt - fit.predictions)))
print(f'Max |rebuilt - prediction| = {max_diff:.2e} (should be ~0)')